# Orbit Wars — Colab Training Pipeline

GPU-accelerated RL training on Google Colab with persistent storage via Google Drive.

**Workflow:**
1. Run the Setup cell (installs deps, mounts Drive, clones repo)
2. Edit the Config cell to choose your experiment
3. Run Training
4. Monitor with TensorBoard, evaluate, generate submission

All checkpoints and logs are saved to Google Drive so they persist across sessions.

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

# ── Clone or update repo ────────────────────────────────────────────────────
REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT THIS
REPO_DIR = '/content/orbit_wars'

if os.path.exists(REPO_DIR):
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# ── Install dependencies ────────────────────────────────────────────────────
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')
os.system('pip install -q "stable-baselines3[extra]>=2.3" gymnasium pyyaml')

print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification

import torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected! Training will be slow.')
    print('Go to Runtime > Change runtime type > GPU')
    DEVICE = 'cpu'

print(f'\nUsing device: {DEVICE}')

## Configuration

Edit the cell below to configure your experiment. You can either:
- Load a YAML config file and apply overrides
- Define the full config as a Python dict

In [ ]:
#@title 3. Experiment Config

from training.train import load_config, apply_dotted_overrides

# Option A: Load from YAML + overrides (recommended)
config = load_config('configs/ppo_default.yaml')
config = apply_dotted_overrides(config, [
    'training.total_timesteps=500000',
    'training.device=cuda',
    'env.n_envs=8',
    # Add more overrides here:
    # 'training.learning_rate=1e-4',
    # 'training.net_arch=[512,512]',
    # 'env.opponent=self',
])

# Option B: Full config dict (uncomment to use instead)
# config = {
#     'env': {'opponent': 'baseline', 'reward_shaping': True, 'n_envs': 8},
#     'training': {
#         'total_timesteps': 1_000_000,
#         'learning_rate': 3e-4,
#         'n_steps': 512,
#         'batch_size': 256,
#         'n_epochs': 10,
#         'gamma': 0.995,
#         'gae_lambda': 0.95,
#         'clip_range': 0.2,
#         'ent_coef': 0.01,
#         'net_arch': [256, 256],
#         'device': 'cuda',
#     },
#     'eval': {'freq': 10_000, 'n_episodes': 20, 'opponent': 'baseline'},
#     'output': {'run_name': 'ppo_colab'},
# }

# Point outputs to Google Drive for persistence
config.setdefault('output', {})
config['output']['checkpoint_dir'] = f'{DRIVE_OUTPUT}/checkpoints'
config['output']['log_dir'] = f'{DRIVE_OUTPUT}/logs'
config['output']['run_name'] = config['output'].get('run_name', 'ppo_colab')

print('Config:')
import yaml
print(yaml.dump(config, default_flow_style=False))

In [ ]:
#@title 4. Train

from training.train import train

# Set resume_from to continue from a previous checkpoint:
# resume_from = f'{DRIVE_OUTPUT}/checkpoints/ppo_colab_20260503_120000/best_model.zip'
resume_from = None

model, checkpoint_dir = train(config, resume_from=resume_from)
print(f'\nCheckpoints saved to: {checkpoint_dir}')

## Experiment Sweep

Run multiple experiments with different hyperparameters. Results are saved to separate subdirectories on Drive.

In [ ]:
#@title 5. Hyperparameter Sweep (optional)

from itertools import product
from training.train import load_config, apply_dotted_overrides, train

# Define sweep parameters
sweep = {
    'training.learning_rate': [3e-4, 1e-4],
    'training.ent_coef': [0.01, 0.005],
}

# Reduce timesteps per experiment for faster sweep
SWEEP_TIMESTEPS = 200_000

keys = list(sweep.keys())
values = list(sweep.values())
results = []

for combo in product(*values):
    overrides = [f'{k}={v}' for k, v in zip(keys, combo)]
    run_label = '_'.join(f'{k.split(".")[-1]}={v}' for k, v in zip(keys, combo))
    print(f'\n{"="*60}\nSweep: {run_label}\n{"="*60}')

    cfg = load_config('configs/ppo_default.yaml')
    cfg = apply_dotted_overrides(cfg, [
        f'training.total_timesteps={SWEEP_TIMESTEPS}',
        'training.device=cuda',
        'env.n_envs=8',
        f'output.run_name=sweep_{run_label}',
    ] + overrides)
    cfg.setdefault('output', {})
    cfg['output']['checkpoint_dir'] = f'{DRIVE_OUTPUT}/checkpoints'
    cfg['output']['log_dir'] = f'{DRIVE_OUTPUT}/logs'

    _, ckpt_dir = train(cfg)
    results.append((run_label, ckpt_dir))

print(f'\n\nSweep complete! {len(results)} experiments ran.')
for label, path in results:
    print(f'  {label} -> {path}')

## Monitoring

In [ ]:
#@title 6. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs

## Evaluation

In [ ]:
#@title 7. Evaluate Trained Model

from pathlib import Path
from agents.rl_agent import RLAgent
from evaluation.evaluate import benchmark

# Use the latest training run, or specify a path:
model_path = checkpoint_dir / 'best_model.zip'
# model_path = Path(f'{DRIVE_OUTPUT}/checkpoints/ppo_colab_XXXXXXXX_XXXXXX/best_model.zip')

if model_path.exists():
    rl_agent = RLAgent(model_path)
    benchmark(rl_agent, agent_name='rl_colab', n_games=20)
else:
    print(f'Model not found: {model_path}')
    print('Available checkpoints on Drive:')
    for p in sorted(Path(f'{DRIVE_OUTPUT}/checkpoints').glob('*/best_model.zip')):
        print(f'  {p}')

## Submission

In [ ]:
#@title 8. Generate Submission

from agents.rl_agent import export_submission

submission_path = f'{DRIVE_OUTPUT}/submissions/submission_rl.py'

if model_path.exists():
    export_submission(model_path, submission_path, mode='rl')
    print(f'Submission saved to: {submission_path}')
    print(f'\nDownload from Google Drive or upload directly:')
    print(f'  kaggle competitions submit orbit-wars -f "{submission_path}" -m "Colab PPO"')
else:
    print('No model to export. Train first (cell 4).')